# 04 Narrative JSON Evaluation

## Objective

This notebook evaluates the LLM-generated DMP narrative JSON outputs created by the Llama-based narrative extraction pipeline.

The goal is to compare the generated narrative text against reference text files and calculate simple content preservation metrics:

- Word Capture
- ROUGE-L
- Word Precision
- Word Recall
- Word F1
- Extracted word count
- Reference word count
- Missing word count
- Extra word count

This notebook focuses only on the narrative answer text, not the full RDA/DMPTool metadata structure.

In [ ]:
from pathlib import Path
import json
import re
import unicodedata
from collections import Counter
from difflib import SequenceMatcher
import pandas as pd

## Step 1: Locate the Project Root

This cell automatically finds the main `dmpbridge` project folder whether the notebook is opened from the project root or from the `notebooks/` folder.

In [ ]:
cwd = Path.cwd()

if (cwd / "data").exists() and (cwd / "src").exists():
    project_root = cwd
else:
    project_root = cwd.parent

print("Project root:", project_root)
print("Data folder exists:", (project_root / "data").exists())

## Step 2: Define Input and Output Folders

The evaluation compares:

- LLM-generated narrative JSON files from `data/llama_narrative_json/`
- Reference text files from `data/reference_text/`

The final evaluation CSV will be saved to `outputs/reports/`.

In [ ]:
llama_narrative_dir = project_root / "data" / "llama_narrative_json"
reference_text_dir = project_root / "data" / "reference_text"
reports_dir = project_root / "outputs" / "reports"

reports_dir.mkdir(parents=True, exist_ok=True)

print("LLM narrative JSON folder exists:", llama_narrative_dir.exists())
print("Reference text folder exists:", reference_text_dir.exists())
print("Reports folder:", reports_dir)

## Step 3: Text Normalization Functions

These functions clean text only for evaluation. They normalize punctuation, whitespace, and ligatures so that small formatting differences do not unfairly reduce the score.

Stopwords are **not removed** because we want a full word-level comparison.

In [ ]:
def normalize_eval_text(text: str) -> str:
    """Normalize text for fair word-level evaluation."""
    if not text:
        return ""

    text = unicodedata.normalize("NFKD", text)

    # Fix common PDF ligature artifacts.
    text = text.replace("ﬁ", "fi").replace("ﬂ", "fl")
    text = text.replace("ﬀ", "ff").replace("ﬃ", "ffi").replace("ﬄ", "ffl")

    # Normalize quotes and dashes.
    text = text.replace("“", '"').replace("”", '"')
    text = text.replace("‘", "'").replace("’", "'")
    text = text.replace("–", "-").replace("—", "-")

    # Lowercase for matching.
    text = text.lower()

    # Keep words/numbers. Replace other punctuation with spaces.
    text = re.sub(r"[^a-z0-9]+", " ", text)

    # Normalize whitespace.
    text = re.sub(r"\s+", " ", text).strip()

    return text


def tokenize_words(text: str) -> list[str]:
    """Tokenize normalized text into words."""
    normalized = normalize_eval_text(text)
    return normalized.split() if normalized else []

## Step 4: Extract Narrative Answer Text from JSON

This function extracts only the narrative answers from the generated DMPTool-compatible JSON.

It looks inside:

```text
narrative → template → section → question → answer → json → answer
```

In [ ]:
def extract_answers_from_narrative_json(narrative_json: dict) -> str:
    """Extract all answer text from the DMPTool-compatible narrative JSON."""
    answers = []

    sections = (
        narrative_json
        .get("narrative", {})
        .get("template", {})
        .get("section", [])
    )

    for section in sections:
        questions = section.get("question", [])

        for question in questions:
            answer_text = (
                question
                .get("answer", {})
                .get("json", {})
                .get("answer", "")
            )

            if answer_text and answer_text != "Not answered":
                answers.append(answer_text.strip())

    return "\n".join(answers)

## Step 5: Evaluation Metrics

This notebook uses simple word-level metrics that are easy to explain in the DMPBridge project.

### Word Capture

Measures how many reference words were captured in the extracted narrative.

### Word Precision

Measures how many extracted words are correct.

### Word Recall

Measures how many reference words were recovered.

### Word F1

Harmonic mean of word precision and word recall.

### ROUGE-L

Measures the longest common subsequence between reference and extracted words. It rewards correct ordering of words.

In [ ]:
def word_counter_overlap(reference_words: list[str], extracted_words: list[str]) -> int:
    """Count overlapping words using frequency-aware matching."""
    ref_counter = Counter(reference_words)
    ext_counter = Counter(extracted_words)
    overlap = ref_counter & ext_counter
    return sum(overlap.values())


def rouge_l_score(reference_words: list[str], extracted_words: list[str]) -> float:
    """Compute a simple ROUGE-L F1 score based on longest common subsequence."""
    if not reference_words or not extracted_words:
        return 0.0

    matcher = SequenceMatcher(None, reference_words, extracted_words)
    lcs = sum(block.size for block in matcher.get_matching_blocks())

    recall = lcs / len(reference_words) if reference_words else 0.0
    precision = lcs / len(extracted_words) if extracted_words else 0.0

    if precision + recall == 0:
        return 0.0

    return (2 * precision * recall) / (precision + recall)


def evaluate_text_pair(reference_text: str, extracted_text: str) -> dict:
    """Evaluate extracted narrative text against reference text."""
    reference_words = tokenize_words(reference_text)
    extracted_words = tokenize_words(extracted_text)

    correct_words = word_counter_overlap(reference_words, extracted_words)

    reference_word_count = len(reference_words)
    extracted_word_count = len(extracted_words)

    missing_word_count = max(reference_word_count - correct_words, 0)
    extra_word_count = max(extracted_word_count - correct_words, 0)

    word_capture = correct_words / reference_word_count if reference_word_count else 0.0
    word_precision = correct_words / extracted_word_count if extracted_word_count else 0.0
    word_recall = correct_words / reference_word_count if reference_word_count else 0.0

    if word_precision + word_recall == 0:
        word_f1 = 0.0
    else:
        word_f1 = 2 * word_precision * word_recall / (word_precision + word_recall)

    rouge_l = rouge_l_score(reference_words, extracted_words)

    return {
        "word_capture": word_capture,
        "rouge_l": rouge_l,
        "word_precision": word_precision,
        "word_recall": word_recall,
        "word_f1": word_f1,
        "extracted_word_count": extracted_word_count,
        "reference_word_count": reference_word_count,
        "missing_word_count": missing_word_count,
        "extra_word_count": extra_word_count,
    }

## Step 6: Evaluate One Sample First

Start with one sample to make sure the file paths and JSON structure are correct.

In [ ]:
sample_name = "sample1"

narrative_json_path = llama_narrative_dir / f"{sample_name}_llama_narrative.json"
reference_path = reference_text_dir / f"{sample_name}_reference.txt"

print("Narrative JSON exists:", narrative_json_path.exists())
print("Reference text exists:", reference_path.exists())

with open(narrative_json_path, "r", encoding="utf-8") as f:
    narrative_json = json.load(f)

reference_text = reference_path.read_text(encoding="utf-8")
extracted_text = extract_answers_from_narrative_json(narrative_json)

metrics = evaluate_text_pair(reference_text, extracted_text)

print(json.dumps(metrics, indent=2))

## Step 7: Inspect Extracted Narrative Text

This helps verify whether the LLM-generated narrative JSON contains the correct answer text.

In [ ]:
print(extracted_text[:3000])

## Step 8: Batch Evaluation for All Samples

This cell evaluates all Llama narrative JSON files that have matching reference text files.

In [ ]:
results = []

for narrative_json_path in sorted(llama_narrative_dir.glob("*_llama_narrative.json")):
    sample_name = narrative_json_path.name.replace("_llama_narrative.json", "")
    reference_path = reference_text_dir / f"{sample_name}_reference.txt"

    if not reference_path.exists():
        print(f"Skipping {sample_name}: missing reference file")
        continue

    with open(narrative_json_path, "r", encoding="utf-8") as f:
        narrative_json = json.load(f)

    extracted_text = extract_answers_from_narrative_json(narrative_json)
    reference_text = reference_path.read_text(encoding="utf-8")

    metrics = evaluate_text_pair(reference_text, extracted_text)
    metrics["sample"] = sample_name

    results.append(metrics)

results_df = pd.DataFrame(results)

if not results_df.empty:
    cols = ["sample"] + [c for c in results_df.columns if c != "sample"]
    results_df = results_df[cols]

results_df

## Step 9: Save Evaluation Results

The final CSV can be used in reports, presentations, or later comparison with the rule-based or PDFPlumber-only outputs.

In [ ]:
evaluation_output_path = reports_dir / "llama_narrative_json_evaluation.csv"

results_df.to_csv(evaluation_output_path, index=False)

print("Saved evaluation results:", evaluation_output_path)

## Step 10: Summary Statistics

This gives an overall summary of how well the Llama-generated narrative JSON preserved the reference content.

In [ ]:
if not results_df.empty:
    metric_columns = [
        "word_capture",
        "rouge_l",
        "word_precision",
        "word_recall",
        "word_f1",
        "extracted_word_count",
        "reference_word_count",
        "missing_word_count",
        "extra_word_count",
    ]

    summary_df = results_df[metric_columns].agg(["mean", "std", "min", "max"]).round(4)
    summary_df
else:
    print("No results to summarize.")

## Interpretation Notes

- High `word_capture` means most reference words were captured.
- High `word_precision` means the generated narrative did not include much extra text.
- High `word_recall` means the generated narrative recovered most of the reference content.
- High `word_f1` means there is a good balance between precision and recall.
- High `rouge_l` means the extracted text preserved the order of the reference text well.

If `word_capture` is high but `rouge_l` is low, the model captured the right words but changed the order or structure.

If `extra_word_count` is high, the model may have included administrative metadata, duplicated text, or hallucinated content.